# TAMER OCR v2.1 — Colab / Kaggle Training Notebook

This notebook is an **execution layer only**. All logic lives in `tamer_ocr/`.

**Pipeline (strict order — no training until ALL data is preprocessed):**
1. Authenticate (HuggingFace + Kaggle)
2. Install dependencies
3. Load codebase
4. Preprocess ALL 4 datasets → verify → push to HF dataset repo
5. Train (auto-resumes from latest checkpoint if interrupted)
6. Push final model to HF

Works on **both** Google Colab and Kaggle Notebooks.

## 1. Detect Environment

In [18]:
import os

# Detect environment
IS_COLAB = os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    print('Environment: Google Colab')
elif IS_KAGGLE:
    print('Environment: Kaggle')
else:
    print('Environment: Unknown (local?)')

# Check GPU
!nvidia-smi || echo 'No GPU found'

Environment: Google Colab
Tue Apr 14 05:17:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

## 2. Authenticate — HuggingFace & Kaggle

In [19]:
import getpass
import os

# --- HuggingFace Token ---
hf_token = os.getenv('HF_TOKEN', '')
if not hf_token:
    hf_token = getpass.getpass('Enter your HuggingFace token (https://huggingface.co/settings/tokens): ')
os.environ['HF_TOKEN'] = hf_token

# --- Kaggle Credentials ---
kaggle_username = 'merselfares'  # Hardcoded as requested
kaggle_key = os.getenv('KAGGLE_KEY', '')
if not kaggle_key:
    kaggle_key = getpass.getpass('Enter your Kaggle API Key (https://www.kaggle.com/settings/account): ')
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

# Write kaggle.json for the CLI
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

# Login to HuggingFace
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=True)

# Get HF username for auto repo naming
from huggingface_hub import HfApi
hf_api = HfApi(token=hf_token)
hf_username = hf_api.whoami()['name']
print(f'Authenticated as: {hf_username} (HuggingFace), {kaggle_username} (Kaggle)')

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Authenticated as: JJKK1212 (HuggingFace), merselfares (Kaggle)


## 3. Install Dependencies

In [20]:
!pip install -q timm albumentations datasets huggingface_hub requests tqdm psutil opencv-python-headless

## 4. Load Codebase

In [21]:
import os, sys

REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"
REPO_DIR = "AI_PROJECT_TAMER_Complete"

# Auto-detect platform
if os.path.isdir("/kaggle/working"):
    WORK_DIR = "/kaggle/working"
else:
    WORK_DIR = "/content"

os.chdir(WORK_DIR)

if os.path.isdir(REPO_DIR):
    print("Repo exists, pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL}

sys.path.insert(0, os.path.join(WORK_DIR, REPO_DIR))
print(f"Codebase ready at {os.path.join(WORK_DIR, REPO_DIR)}")

Repo exists, pulling latest...
Already up to date.
Codebase ready at /kaggle/working/AI_PROJECT_TAMER_Complete


In [22]:
import sys
import os

# Add the repo root to Python path so tamer_ocr is importable
# Adjust the path below to match where you cloned the repo
# Typical Colab: /content/tamer_ocr_v2.1  or  /content/TAMER-OCR
# Typical Kaggle: /kaggle/working/tamer_ocr_v2.1

# Auto-detect: find the directory containing tamer_ocr/
for candidate in ["/content", "/kaggle/working"]:
    for sub in os.listdir(candidate):
        pkg_path = os.path.join(candidate, sub, "tamer_ocr")
        if os.path.isdir(pkg_path):
            repo_root = os.path.join(candidate, sub)
            if repo_root not in sys.path:
                sys.path.insert(0, repo_root)
                print(f"Added to sys.path: {repo_root}")
            break

# Fallback: if auto-detect failed, set manually
if not any("tamer_ocr" in p for p in sys.path):
    # Change this to your actual clone path
    manual_path = "/content/TAMER-OCR"  # <-- adjust if needed
    if os.path.isdir(os.path.join(manual_path, "tamer_ocr")):
        sys.path.insert(0, manual_path)
        print(f"Added manually: {manual_path}")
    else:
        print("WARNING: Could not find tamer_ocr package!")
        print(f"sys.path entries: {sys.path}")

sys.path entries: ['/kaggle/working/AI_PROJECT_TAMER_Complete', '/kaggle/working/AI_PROJECT_TAMER_Complete', '/kaggle/working', '/kaggle/lib/kagglegym', '/kaggle/lib', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']


## 5. Configure Training

In [23]:
from tamer_ocr.config import Config

config = Config()

# === PATHS ===
if os.path.exists('/kaggle'):
    config.data_dir = '/kaggle/working/data'
    config.output_dir = '/kaggle/working/outputs'
    config.checkpoint_dir = '/kaggle/working/checkpoints'
    config.log_dir = '/kaggle/working/logs'
else:
    config.data_dir = '/content/data'
    config.output_dir = '/content/outputs'
    config.checkpoint_dir = '/content/checkpoints'
    config.log_dir = '/content/logs'

# === HF REPOS ===
config.hf_token = hf_token
config.hf_repo_id = f"{hf_username}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{hf_username}/tamer-preprocessed"

# === KAGGLE ===
config.kaggle_username = 'merselfares'
config.kaggle_key = kaggle_key

# === MODEL (defaults are correct) ===
# config.encoder_name = 'swin_base_patch4_window7_224'
# config.d_model = 512
# config.num_decoder_layers = 6

# === TRAINING ===
config.batch_size = 8
config.accumulation_steps = 4  # Effective batch = 32
config.num_epochs = 150
config.encoder_lr = 1e-5
config.decoder_lr = 1e-4
config.label_smoothing = 0.1
config.total_training_steps = 50000

# === CHECKPOINTING ===
config.checkpoint_every_epochs = 3  # Save every 3 epochs
config.keep_last_n_checkpoints = 3

# === INFERENCE ===
config.beam_width = 5
config.length_penalty = 0.6

print('Configuration ready!')
print(f'  HF model repo: {config.hf_repo_id}')
print(f'  HF dataset repo: {config.hf_dataset_repo_id}')
print(f'  Checkpoint dir: {config.checkpoint_dir}')
print(f'  Effective batch size: {config.batch_size * config.accumulation_steps}')
print(f'  Checkpoint every: {config.checkpoint_every_epochs} epochs')

Configuration ready!
  HF model repo: JJKK1212/tamer-ocr-model
  HF dataset repo: JJKK1212/tamer-preprocessed
  Checkpoint dir: /kaggle/working/checkpoints
  Effective batch size: 32
  Checkpoint every: 3 epochs


In [24]:
from tamer_ocr.config import Config
import dataclasses
c = Config()
for f in dataclasses.fields(c):
    print(f"{f.name} = {getattr(c, f.name)}")

data_root = /content/tamer_data
data_dir = ./data
output_dir = ./outputs
checkpoint_dir = ./checkpoints
log_dir = ./logs
datasets = [{'name': 'crohme', 'type': 'kaggle', 'kaggle_slug': 'nguyenvo0921/crohme', 'parser': 'crohme'}, {'name': 'hme100k', 'type': 'kaggle', 'kaggle_slug': 'muhammadhanif/hme100k', 'parser': 'hme100k'}, {'name': 'im2latex', 'type': 'kaggle', 'kaggle_slug': 'shahrukhkhan/im2latex', 'parser': 'im2latex'}, {'name': 'mathwriting', 'type': 'huggingface', 'hf_repo': 'MathInstruct/MathWriting', 'parser': 'crohme'}]
auto_download = False
skip_validation = False
img_height = 256
img_width = 1024
max_token_length = 150
max_aspect_ratio = 10.0
encoder_name = swin_base_patch4_window7_224
d_model = 512
nhead = 8
num_decoder_layers = 6
dim_feedforward = 2048
dropout = 0.1
encoder_feature_dim = 1024
batch_size = 8
accumulation_steps = 4
num_workers = 2
num_epochs = 150
encoder_lr = 1e-05
decoder_lr = 0.0001
weight_decay = 0.0001
max_grad_norm = 1.0
label_smoothing = 0.1
temp_s

## 6. Run Full Pipeline

**This runs the strict pipeline:**
1. Preprocess ALL datasets
2. Verify → push to HF dataset repo
3. Build model
4. Auto-resume from latest checkpoint (if interrupted)
5. Train
6. Final beam search evaluation

If your session crashes, just **re-run all cells** — it will auto-resume from the last checkpoint.

In [25]:
import os
data_dir = "/kaggle/working/AI_PROJECT_TAMER_Complete/tamer_ocr/data"
print("Files in tamer_ocr/data/:")
for f in sorted(os.listdir(data_dir)):
    print(f"  {f}")

Files in tamer_ocr/data/:
  __init__.py
  __pycache__
  advanced_downloader.py
  augmentation.py
  data_manager.py
  dataset.py
  datasets_registry.py
  downloader.py
  parser.py
  sampler.py
  tokenizer.py
  validator.py


In [26]:
import os
data_dir = "/kaggle/working/AI_PROJECT_TAMER_Complete/tamer_ocr/data"
for f in os.listdir(data_dir):
    if f.endswith(".py"):
        path = os.path.join(data_dir, f)
        with open(path) as fh:
            content = fh.read()
        if "LaTeXNormalizer" in content or "latex_normal" in content.lower():
            print(f"  Found in: {f}")

  Found in: data_manager.py
  Found in: __init__.py


In [27]:
from tamer_ocr.core.trainer import Trainer

trainer = Trainer(config)
trainer.run()  # All logic inside — auto-resumes if interrupted

2026-04-14 05:17:53 - TAMER.Trainer - INFO - Device: cuda
2026-04-14 05:17:53 - TAMER.Trainer - INFO - GPU: Tesla T4
2026-04-14 05:17:53 - TAMER.Trainer - INFO - VRAM: 15.6 GB
2026-04-14 05:17:53 - TAMER.Trainer - INFO - ======================================================================
2026-04-14 05:17:53 - TAMER.Trainer - INFO - PHASE 1: Data Preprocessing (STRICT — no training until complete)
2026-04-14 05:17:53 - TAMER.Trainer - INFO - ======================================================================
STARTING STRICT PREPROCESSING PIPELINE

[STEP 1/5] Downloading datasets...

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
  ✓ crohme downloaded

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
  ✓ hme100k downloaded

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata
  ✓ im2latex downloaded

    HF

ModuleNotFoundError: No module named 'tamer_ocr.data.latex_normalizer'

## 7. Push Final Model to HuggingFace

In [ ]:
import os
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf

best_path = os.path.join(config.checkpoint_dir, 'best.pt')
if os.path.exists(best_path):
    push_checkpoint_to_hf(best_path, config, epoch=0, is_best=True)
    print(f'Pushed best.pt to {config.hf_repo_id}')
else:
    print('No best.pt found — training may not have completed.')

## 8. Beam Search Evaluation (Optional)

Run a full beam search eval on the best checkpoint.

In [ ]:
import os
from tamer_ocr.utils.checkpoint import find_latest_checkpoint

best_path = os.path.join(config.checkpoint_dir, 'best.pt')
if not os.path.exists(best_path):
    best_path = find_latest_checkpoint(config.checkpoint_dir)

if best_path:
    trainer = Trainer(config)
    trainer.preprocess_data()
    trainer.create_dataloaders()
    trainer.build_model()
    trainer.resume_from_checkpoint(best_path)
    metrics = trainer.evaluate_with_beam_search(max_samples=500)
    print('\nBeam Search Results:')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')
else:
    print('No checkpoint found for evaluation.')